In [1]:
from pyspark.sql import SparkSession
from pyspark import SparkConf

In [21]:
sp_conf = SparkConf() 
sp_conf.set("spark.sql.catalog.glue_catalog", "org.apache.iceberg.spark.SparkCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.warehouse", "s3://iceberg-wh-east/")
sp_conf.set("spark.sql.catalog.glue_catalog.catalog-impl", "org.apache.iceberg.aws.glue.GlueCatalog")
sp_conf.set("spark.sql.catalog.glue_catalog.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
sp_conf.set("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
sp_conf.set("spark.hadoop.fs.s3a.aws.credentials.provider","com.amazonaws.auth.DefaultAWSCredentialsProviderChain")
sp_conf.set("spark.sql.defaultCatalog", "glue_catalog")
spark.sparkContext._jsc.hadoopConfiguration().set("fs.s3a.aws.credentials.provider", "com.amazonaws.auth.DefaultAWSCredentialsProviderChain")

In [22]:
spark = SparkSession.builder \
    .appName("Glue-Iceberg-Integration") \
    .config(conf=sp_conf) \
    .getOrCreate()

In [4]:
spark.sql("""
    CREATE DATABASE IF NOT EXISTS glue_catalog.berg 
""")

DataFrame[]

In [17]:
spark.sql("""
    CREATE TABLE IF NOT EXISTS glue_catalog.berg.orders (
        order_id BIGINT,
        customer_id BIGINT,
        order_amount DECIMAL(10,2),
        order_ts TIMESTAMP
    )
    USING iceberg PARTITIONED BY (HOUR(order_ts))
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS glue_catalog.berg.orders_staging (
        order_id BIGINT,
        customer_id BIGINT,
        order_amount DECIMAL(10,2),
        order_ts TIMESTAMP
    )
    USING iceberg PARTITIONED BY (HOUR(order_ts))
""")

DataFrame[]

In [27]:
spark.sql("select * from berg.orders").show()

[Stage 33:>                                                         (0 + 1) / 1]

+--------+-----------+------------+--------------------+
|order_id|customer_id|order_amount|            order_ts|
+--------+-----------+------------+--------------------+
|     123|        456|       36.17|2025-08-07 15:53:...|
|     124|        456|       36.17|2025-08-07 15:54:...|
+--------+-----------+------------+--------------------+



In [19]:
spark.sql("""insert into glue_catalog.berg.orders values( 123, 456, 36.17, CURRENT_TIMESTAMP())""")
spark.sql("""insert into glue_catalog.berg.orders_staging values( 123, 456, 36.17, CURRENT_TIMESTAMP())""")
spark.sql("""insert into glue_catalog.berg.orders_staging values( 124, 456, 36.17, CURRENT_TIMESTAMP())""")

DataFrame[]

In [26]:
spark.sql("""MERGE INTO berg.orders o
USING (SELECT * FROM berg.orders_staging) s
ON o.order_id = s.order_id
WHEN MATCHED THEN UPDATE SET order_amount = s.order_amount
WHEN NOT MATCHED THEN INSERT *;
""")

DataFrame[]

In [32]:
spark.sql("SELECT * FROM berg.orders.history").show(truncate=False)

+-----------------------+-------------------+-------------------+-------------------+
|made_current_at        |snapshot_id        |parent_id          |is_current_ancestor|
+-----------------------+-------------------+-------------------+-------------------+
|2025-08-07 15:53:54.177|1316513614785652826|NULL               |true               |
|2025-08-07 16:02:12.269|1220284206999610676|1316513614785652826|true               |
+-----------------------+-------------------+-------------------+-------------------+



In [33]:
spark.sql("SELECT file, latest_snapshot_id FROM berg.orders.metadata_log_entries").show(truncate=False)

+-----------------------------------------------------------------------------------------------------+-------------------+
|file                                                                                                 |latest_snapshot_id |
+-----------------------------------------------------------------------------------------------------+-------------------+
|s3://iceberg-wh-east/berg.db/orders/metadata/00000-88ebaec1-2d03-43a9-99ab-2feff5b817fe.metadata.json|NULL               |
|s3://iceberg-wh-east/berg.db/orders/metadata/00001-de7af66b-d62e-4e6f-a778-bb8997b5d3e8.metadata.json|1316513614785652826|
|s3://iceberg-wh-east/berg.db/orders/metadata/00002-dfd8ddf3-5e70-4491-b42e-a4fbacd37acf.metadata.json|1220284206999610676|
+-----------------------------------------------------------------------------------------------------+-------------------+



In [20]:
import boto3
def active_iceberg_table_metadata(active_database_name, active_table_name):
    glue = boto3.client("glue", region_name = 'us-east-1')
    table = glue.get_table(DatabaseName=active_database_name, Name=active_table_name)
    parameters = table["Table"]["Parameters"]
    full_path_metadata_location = parameters["metadata_location"]
    return full_path_metadata_location.split('/')[-1]

print(active_iceberg_table_metadata("berg", "icetable1"))   

00006-a98730a9-ef2e-498f-a370-63e2f1794105.metadata.json


In [25]:
latest_metadata = active_iceberg_table_metadata("berg", "icetable1")

# Create a DynamoDB resource
dynamodb = boto3.resource('dynamodb', region_name='us-east-1')

# Get a table resource
table = dynamodb.Table('latest_metadata')

# Define the item to be inserted/updated
item = {
    'dbtable': 'berg.icetable1',
    'metadatafile': latest_metadata,
}

try:
    response = table.put_item(
        Item=item
    )
    print("Item put successfully:", response)
except Exception as e:
    print("Error putting item:", e)


Item put successfully: {'ResponseMetadata': {'RequestId': '5R63COIOLCD3DCSCLGPT51QAMRVV4KQNSO5AEMVJF66Q9ASUAAJG', 'HTTPStatusCode': 200, 'HTTPHeaders': {'server': 'Server', 'date': 'Sun, 03 Aug 2025 01:29:20 GMT', 'content-type': 'application/x-amz-json-1.0', 'content-length': '2', 'connection': 'keep-alive', 'x-amzn-requestid': '5R63COIOLCD3DCSCLGPT51QAMRVV4KQNSO5AEMVJF66Q9ASUAAJG', 'x-amz-crc32': '2745614147'}, 'RetryAttempts': 0}}


In [12]:
def update_iceberg_table_metadata(active_database_name, active_table_name, metadata):
    glue = boto3.client("glue", region_name = 'us-east-1')
    table = glue.get_table(DatabaseName=active_database_name, Name=active_table_name)
    table_input = table["Table"]
    table_input["Parameters"]["metadata_location"] = f"s3://iceberg-wh-east/berg.db/icetable1/metadata/{metadata}"
    
    keys_to_remove = ['CreateTime', 'UpdateTime', 'IsRegisteredWithLakeFormation', 'CatalogId', 'DatabaseName', 'CreatedBy', 'VersionId', 'IsMultiDialectView']
    
    for key in keys_to_remove:
        if key in table_input: del table_input[key]

    print(table_input)
    glue.update_table(
        DatabaseName=active_database_name,
        TableInput=table_input
    )
    return
# update_iceberg_table_metadata("berg", "icetable1", "00004-bafdff20-352d-4b17-89ae-a75eda17bd3c.metadata.json")   

{'Name': 'icetable1', 'Retention': 0, 'StorageDescriptor': {'Columns': [{'Name': 'id', 'Type': 'int', 'Parameters': {'iceberg.field.current': 'true', 'iceberg.field.id': '1', 'iceberg.field.optional': 'true'}}, {'Name': 'name', 'Type': 'string', 'Parameters': {'iceberg.field.current': 'true', 'iceberg.field.id': '2', 'iceberg.field.optional': 'true'}}], 'Location': 's3://iceberg-wh-east/berg.db/icetable1', 'AdditionalLocations': [], 'Compressed': False, 'NumberOfBuckets': 0, 'SortColumns': [], 'StoredAsSubDirectories': False}, 'TableType': 'EXTERNAL_TABLE', 'Parameters': {'metadata_location': 's3://iceberg-wh-east/berg.db/icetable1/metadata/00004-bafdff20-352d-4b17-89ae-a75eda17bd3c.metadata.json', 'previous_metadata_location': 's3://iceberg-wh-east/berg.db/icetable1/metadata/00003-19cbcc94-a339-4934-949d-888a57eeb46f.metadata.json', 'table_type': 'ICEBERG'}}
